In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re 
import glob
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from matplotlib import colors
import matplotlib.ticker as ticker

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.regression.mixed_linear_model import MixedLM
import statsmodels.stats.multitest as smm

/Users/z3344650/opt/anaconda3/lib/python3.8/site-packages/statsmodels/tsa/base/tsa_model.py:7: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import (to_datetime, Int64Index, DatetimeIndex, Period,
/Users/z3344650/opt/anaconda3/lib/python3.8/site-packages/statsmodels/tsa/base/tsa_model.py:7: FutureWarning: pandas.Float64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import (to_datetime, Int64Index, DatetimeIndex, Period,


In [2]:
# Read in clustering results
cells = pd.read_csv('../data/patient_metacluster_table.csv')
cells = cells.loc[~cells['Patient_ID'].str.startswith('Mobilised'), :]
cells = cells.loc[~cells['Patient_ID'].str.startswith('PBMC_Control'), :]

cells.head()

,FileNames,metacluster_1,metacluster_10,metacluster_11,metacluster_12,metacluster_13,metacluster_14,metacluster_15,metacluster_16,metacluster_17,...,Patient_ID,Timepoint,Batch_Control,Reference,Batch_Control_Type,Healthy,Disease,Batch,X7_month_response,X12_month_response
0,61201_001_C1_D8_Myeloid_Panel,9894.0,0.0,2569.0,30246.0,4.0,1323.0,6842.0,5517.0,82.0,...,61201_001,C1_D8,C1_D8,False,not_control,Sample,MDS,8,Non_responder,Progression
1,61201_001_C7_D1_Myeloid_Panel,2411.0,0.0,257.0,2476.0,2.0,429.0,1703.0,754.0,10.0,...,61201_001,C7_D1,C7_D1,False,not_control,Sample,MDS,8,Non_responder,Progression
2,61201_001_C7_D22_Myeloid_Panel,454.0,0.0,13.0,114.0,0.0,35.0,207.0,469.0,0.0,...,61201_001,C7_D22,C7_D22,False,not_control,Sample,MDS,8,Non_responder,Progression
3,61201_001_SPD_Myeloid_Panel,373.0,0.0,14.0,146.0,2.0,50.0,83.0,320.0,2.0,...,61201_001,SPD,SPD,False,not_control,Sample,MDS,8,Non_responder,Progression
4,61201_002_C1_D1_Myeloid_Panel,18476.0,8.0,184.0,2058.0,106.0,180.0,2503.0,525.0,0.0,...,61201_002,C1_D1,C1_D1,False,not_control,Sample,AML,1,Non_responder,Off_trial_at_month_7


In [3]:
# Create alias dictionaries
alias = pd.read_excel('../data/patient_alias_from_2025.xlsx')
alias_dict = dict(zip(alias['PID'].astype(str), alias['alias']))

control = pd.read_csv('../data/healthy_control.csv')
control_dict = dict(zip(control['key'].str.replace('_T_Cell_Panel.fcs', ''), control['value'].str.replace('.fcs', '', regex=False)))
control_dict = {k.replace('Healthy_BMA_67y_Male', 'Healthy_BMA_67Y_Male'): v for k, v in control_dict.items()}

<ipython-input-3-1ab1d6a91885>:6: FutureWarning: The default value of regex will change from True to False in a future version.
  control_dict = dict(zip(control['key'].str.replace('_T_Cell_Panel.fcs', ''), control['value'].str.replace('.fcs', '', regex=False)))


In [4]:
# Replace the patient identifiers in the file with our agreed alias
pid_list = []
for i in cells['Patient_ID']:
    if i.startswith('6'):
        x = alias_dict[i.replace('_', '')]
        pid_list.append(x)
    elif i.startswith('He'):
        x = control_dict[i]
        pid_list.append(x)
        
cells['Patient_ID'] = pid_list    

In [5]:
cells.head()

,FileNames,metacluster_1,metacluster_10,metacluster_11,metacluster_12,metacluster_13,metacluster_14,metacluster_15,metacluster_16,metacluster_17,...,Patient_ID,Timepoint,Batch_Control,Reference,Batch_Control_Type,Healthy,Disease,Batch,X7_month_response,X12_month_response
0,61201_001_C1_D8_Myeloid_Panel,9894.0,0.0,2569.0,30246.0,4.0,1323.0,6842.0,5517.0,82.0,...,P08,C1_D8,C1_D8,False,not_control,Sample,MDS,8,Non_responder,Progression
1,61201_001_C7_D1_Myeloid_Panel,2411.0,0.0,257.0,2476.0,2.0,429.0,1703.0,754.0,10.0,...,P08,C7_D1,C7_D1,False,not_control,Sample,MDS,8,Non_responder,Progression
2,61201_001_C7_D22_Myeloid_Panel,454.0,0.0,13.0,114.0,0.0,35.0,207.0,469.0,0.0,...,P08,C7_D22,C7_D22,False,not_control,Sample,MDS,8,Non_responder,Progression
3,61201_001_SPD_Myeloid_Panel,373.0,0.0,14.0,146.0,2.0,50.0,83.0,320.0,2.0,...,P08,SPD,SPD,False,not_control,Sample,MDS,8,Non_responder,Progression
4,61201_002_C1_D1_Myeloid_Panel,18476.0,8.0,184.0,2058.0,106.0,180.0,2503.0,525.0,0.0,...,P24,C1_D1,C1_D1,False,not_control,Sample,AML,1,Non_responder,Off_trial_at_month_7


In [6]:
# Replace the patient identifiers in the FileName column with our agreed alias
file_list = []
for i in cells['FileNames']:
    if i.startswith('6'):
        x = i.split('_')[0] + i.split('_')[1] 
        x = alias_dict[x] + '_'
        y = '_'.join(i.split('_')[2:])
        z = x + y
        file_list.append(z)
    elif i.startswith('Healthy'):
        x = i.replace('_Myeloid_Panel', '')
        x = control_dict[x] + '_Myeloid_Panel'
        file_list.append(x)
        
cells['FileNames'] = file_list

In [10]:
cells.to_csv('../data/patient_metacluster_table_patient_updated.csv')

In [9]:
for i in cells['FileNames'].unique():
    if i.startswith('6'):
        print(i)